# Load `oil_network` from `asset_graph.json`

Populates the `oil_network` schema from `asset_graph/asset_graph.json`. Loads:

- **commodities** — single row, `crude`
- **graphs** — single row, `us_crude_starter`
- **locations** — one per asset (lat/lon if present; otherwise just labels)
- **assets** — 74 rows. `node_class`, `node_subtype`, `configuration`, `resolution_hierarchy`, `kind` (physical vs abstract), `starter_status` all packed into `assets.attributes` JSONB
- **node_types** — distinct `node_subtype` values
- **nodes** — 74 rows attaching each asset to the starter graph, with `node_type` set
- **variables** — 4 non-relational slots per node (`production`, `consumption`, `inventory`, `balancing_item`) plus an `inflow`/`outflow` pair per non-aggregation edge (deduped by `(source, target)`)
- **scenarios** — single row, `starter_us_crude_2015_2025`. The full contract metadata (authoritative levels, collapsed nodes, time range) is preserved as `scenarios.attributes`

**Idempotent.** Every insert is `ON CONFLICT ... DO UPDATE`, so re-running this notebook updates rows in place. To start from a clean state, re-run [build_oil_network.ipynb](build_oil_network.ipynb) first — it drops and recreates the schema.

**Out of scope here:** `timeseries`, `timeseries_data`, and `variable_assignments`. Those land later when EIA data is fetched and we know which TS feeds which variable. The contract metadata in `scenarios.attributes` is what the eventual assignments-loader will consume.

## 0. Init

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from IPython.display import display

PG_HOST   = os.environ.get("PG_HOST",   "localhost")
PG_PORT   = os.environ.get("PG_PORT",   "5432")
PG_DB     = os.environ.get("PG_DB",     "eia_crude")
PG_USER   = os.environ.get("PG_USER",   "eia_user")
PG_PASS   = os.environ.get("PG_PASS",   "eia_password")
PG_SCHEMA = os.environ.get("PG_SCHEMA", "oil_network")
PG_URL    = f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"

from paths import ASSET_GRAPH_JSON
JSON_PATH = ASSET_GRAPH_JSON

GRAPH_ID    = "us_crude_starter"
SCENARIO_ID = "starter_us_crude_2015_2025"
COMMODITY   = "crude"

engine = create_engine(
    PG_URL,
    connect_args={"options": f"-csearch_path={PG_SCHEMA},public"},
    future=True,
)

with engine.connect() as c:
    ver, db, usr = c.execute(text("SELECT version(), current_database(), current_user")).one()
print(f"Connected:     {db} as {usr}")
print(f"Server:        {ver.splitlines()[0]}")
print(f"Target schema: {PG_SCHEMA}")
print(f"JSON source:   {JSON_PATH} (exists={JSON_PATH.exists()})")
print(f"Graph:         {GRAPH_ID}")
print(f"Scenario:      {SCENARIO_ID}")
print(f"Commodity:     {COMMODITY}")

Connected:     eia_crude as eia_user
Server:        PostgreSQL 18.3 on x86_64-windows, compiled by msvc-19.44.35223, 64-bit
Target schema: oil_network
JSON source:   C:\Users\PedroPorfirio\OneDrive - Jabuticaba\Oil Network Project\Thesis\clean\config\asset_graph.json (exists=True)
Graph:         us_crude_starter
Scenario:      starter_us_crude_2015_2025
Commodity:     crude


## 1. Read the JSON

In [2]:
with open(JSON_PATH, encoding="utf-8") as f:
    G = json.load(f)

nodes_json    = G["nodes"]
edges_json    = G["edges"]
contract_json = G["starter_coverage_contract"]
meta_json     = G["meta"]

print(f"Loaded {len(nodes_json)} nodes, {len(edges_json)} edges from {JSON_PATH}")
print(f"Edge-type counts: {meta_json['edge_type_counts']}")
print(f"Subtype counts:   {meta_json['node_subtype_counts']}")
print(f"Contract:         {contract_json['name']}")

Loaded 219 nodes, 409 edges from C:\Users\PedroPorfirio\OneDrive - Jabuticaba\Oil Network Project\Thesis\clean\config\asset_graph.json
Edge-type counts: {'gathering': 7, 'shipping_route': 11, 'terminal_connector': 19, 'spr_release': 4, 'spr_fill': 4, 'pipeline_intake': 28, 'pipeline_outflow': 30, 'production_to_gathering': 6, 'production_outflow': 15, 'offshore_outflow': 5, 'refinery_inflow': 243, 'terminal_export': 4, 'foreign_inflow': 6, 'canadian_inflow': 6, 'foreign_outflow': 6, 'inter_padd_region_flow': 12}
Subtype counts:   {'state_sub_basin': 5, 'offshore_region': 1, 'state_conventional': 7, 'observational_aggregate': 5, 'padd_view': 5, 'gathering': 5, 'origin_terminal': 6, 'storage_terminal': 10, 'spr_site': 4, 'export_terminal': 7, 'import_terminal': 4, 'pipeline': 24, 'refining_centre_view': 5, 'refinery': 115, 'foreign_production_aggregate': 1, 'refining_district_view': 10}
Contract:         US_crude_starter_monthly_2015_2025


## 2. Commodities and graph

In [3]:
with engine.begin() as c:
    c.execute(
        text("""
            INSERT INTO commodities(commodity, description)
            VALUES (:commodity, :description)
            ON CONFLICT (commodity) DO UPDATE
              SET description = EXCLUDED.description
        """),
        {"commodity": COMMODITY, "description": "Crude oil (generic)"},
    )

    c.execute(
        text("""
            INSERT INTO graphs(graph_id, name, description, attributes)
            VALUES (:graph_id, :name, :description, CAST(:attributes AS JSONB))
            ON CONFLICT (graph_id) DO UPDATE
              SET name        = EXCLUDED.name,
                  description = EXCLUDED.description,
                  attributes  = EXCLUDED.attributes
        """),
        {
            "graph_id":    GRAPH_ID,
            "name":        "US crude starter graph",
            "description": meta_json.get("description"),
            "attributes":  json.dumps({
                "schema_version": meta_json.get("schema_version"),
                "thesis_version": meta_json.get("thesis_version"),
                "source_file":    str(JSON_PATH),
            }),
        },
    )

with engine.connect() as c:
    print("commodities:", c.execute(text("SELECT count(*) FROM commodities")).scalar())
    print("graphs:     ", c.execute(text("SELECT count(*) FROM graphs")).scalar())

commodities: 1
graphs:      1


## 3. Locations and assets

One location row per asset. Every JSON node becomes one asset; geography → location, configuration + class metadata → `assets.attributes` JSONB. PADDs and other observational aggregates get rows here too — they're abstract assets per our agreed definition (asset = identity, can be physical or abstract; location is just where).

Some pipelines have `padd` as a list (they span multiple PADDs); we store the first as the flat `padd` column and keep the full list under `locations.attributes.padd_list`.

In [4]:
# Abstract vs physical: derive from `node_class` rather than a hardcoded subtype list.
# `observational` -> abstract (PADD views, basin aggregates, refining-centre views, ...).
# `infrastructure` -> physical (terminals, pipelines, refineries, gathering, ...).
# Anything else (none in the current schema) defaults to physical.

location_rows = []
asset_rows    = []

for n in nodes_json:
    nid    = n["id"]
    sub    = n["node_subtype"]
    cls    = n.get("node_class")
    geog   = n.get("geography") or {}
    config = n.get("configuration") or {}
    rh     = n.get("resolution_hierarchy") or {}

    is_abstract = (cls == "observational")

    padd_raw = geog.get("padd")
    padd_flat = padd_raw[0] if isinstance(padd_raw, list) and padd_raw else (padd_raw if not isinstance(padd_raw, list) else None)

    location_id = f"loc__{nid}"
    location_rows.append({
        "location_id": location_id,
        "name":        n["name"],
        "lat":         geog.get("lat"),
        "lon":         geog.get("lon"),
        "country":     geog.get("country"),
        "state":       geog.get("state"),
        "padd":        padd_flat,
        "county":      geog.get("county"),
        "sea":         geog.get("sea"),
        "attributes":  json.dumps({
            "endpoint_intake_refs":  geog.get("endpoint_intake_refs"),
            "endpoint_outflow_refs": geog.get("endpoint_outflow_refs"),
            "spans_multiple_padds":  geog.get("spans_multiple_padds"),
            "padd_list":             padd_raw if isinstance(padd_raw, list) else None,
        }),
    })

    asset_rows.append({
        "asset_id":    nid,
        "name":        n["name"],
        "description": n.get("notes"),
        "location_id": location_id,
        "attributes":  json.dumps({
            "node_class":           cls,
            "node_subtype":         sub,
            "kind":                 "abstract" if is_abstract else "physical",
            "starter_status":       n.get("starter_status"),
            "configuration":        config,
            "resolution_hierarchy": rh,
        }),
    })

with engine.begin() as c:
    c.execute(
        text("""
            INSERT INTO locations(location_id, name, lat, lon, country, state, padd, county, sea, attributes)
            VALUES (:location_id, :name, :lat, :lon, :country, :state, :padd, :county, :sea, CAST(:attributes AS JSONB))
            ON CONFLICT (location_id) DO UPDATE
              SET name = EXCLUDED.name,
                  lat = EXCLUDED.lat,
                  lon = EXCLUDED.lon,
                  country = EXCLUDED.country,
                  state = EXCLUDED.state,
                  padd = EXCLUDED.padd,
                  county = EXCLUDED.county,
                  sea = EXCLUDED.sea,
                  attributes = EXCLUDED.attributes
        """),
        location_rows,
    )
    c.execute(
        text("""
            INSERT INTO assets(asset_id, name, description, location_id, attributes)
            VALUES (:asset_id, :name, :description, :location_id, CAST(:attributes AS JSONB))
            ON CONFLICT (asset_id) DO UPDATE
              SET name        = EXCLUDED.name,
                  description = EXCLUDED.description,
                  location_id = EXCLUDED.location_id,
                  attributes  = EXCLUDED.attributes
        """),
        asset_rows,
    )

with engine.connect() as c:
    print("locations:", c.execute(text("SELECT count(*) FROM locations")).scalar())
    print("assets:   ", c.execute(text("SELECT count(*) FROM assets")).scalar())
    print("  abstract:", c.execute(text("SELECT count(*) FROM assets WHERE attributes->>'kind' = 'abstract'")).scalar())
    print("  physical:", c.execute(text("SELECT count(*) FROM assets WHERE attributes->>'kind' = 'physical'")).scalar())

locations: 219
assets:    219
  abstract: 24
  physical: 195


## 4. Node types

Distinct values of `node_subtype` from the JSON become rows in `node_types`.

In [5]:
node_type_values = sorted({n["node_subtype"] for n in nodes_json})

with engine.begin() as c:
    c.execute(
        text("""
            INSERT INTO node_types(node_type, description)
            VALUES (:node_type, :description)
            ON CONFLICT (node_type) DO UPDATE
              SET description = EXCLUDED.description
        """),
        [
            {"node_type": t, "description": f"Auto-loaded from JSON node_subtype '{t}'."}
            for t in node_type_values
        ],
    )

with engine.connect() as c:
    nt = pd.read_sql(text("SELECT node_type, description FROM node_types ORDER BY node_type"), c)
display(nt)

,node_type,description
0,export_terminal,Auto-loaded from JSON node_subtype 'export_ter...
1,foreign_export_destination,Auto-loaded from JSON node_subtype 'foreign_ex...
2,foreign_production_aggregate,Auto-loaded from JSON node_subtype 'foreign_pr...
3,gathering,Auto-loaded from JSON node_subtype 'gathering'.
4,import_terminal,Auto-loaded from JSON node_subtype 'import_ter...
5,observational_aggregate,Auto-loaded from JSON node_subtype 'observatio...
6,offshore_region,Auto-loaded from JSON node_subtype 'offshore_r...
7,origin_terminal,Auto-loaded from JSON node_subtype 'origin_ter...
8,pipeline,Auto-loaded from JSON node_subtype 'pipeline'.
9,refinery,Auto-loaded from JSON node_subtype 'refinery'.


## 5. Nodes (asset-in-graph junction)

Each JSON node becomes one row in `nodes`, attaching the asset to `us_crude_starter` and labelling it with its `node_type`.

In [6]:
node_rows = [
    {
        "node_id":    n["id"],
        "graph_id":   GRAPH_ID,
        "asset_id":   n["id"],
        "node_type":  n["node_subtype"],
        "notes":      n.get("notes"),
        "attributes": json.dumps({"starter_status": n.get("starter_status")}),
    }
    for n in nodes_json
]

with engine.begin() as c:
    c.execute(
        text("""
            INSERT INTO nodes(node_id, graph_id, asset_id, node_type, notes, attributes)
            VALUES (:node_id, :graph_id, :asset_id, :node_type, :notes, CAST(:attributes AS JSONB))
            ON CONFLICT (node_id) DO UPDATE
              SET graph_id   = EXCLUDED.graph_id,
                  asset_id   = EXCLUDED.asset_id,
                  node_type  = EXCLUDED.node_type,
                  notes      = EXCLUDED.notes,
                  attributes = EXCLUDED.attributes
        """),
        node_rows,
    )

with engine.connect() as c:
    n_in_graph = c.execute(
        text("SELECT count(*) FROM nodes WHERE graph_id = :g"),
        {"g": GRAPH_ID},
    ).scalar()
    by_type = pd.read_sql(text("""
        SELECT node_type, count(*) AS n
        FROM nodes
        WHERE graph_id = :g
        GROUP BY 1 ORDER BY 1
    """), c, params={"g": GRAPH_ID})
print(f"Nodes in {GRAPH_ID}: {n_in_graph}")
display(by_type)

Nodes in us_crude_starter: 219


,node_type,n
0,export_terminal,7
1,foreign_export_destination,1
2,foreign_production_aggregate,2
3,gathering,5
4,import_terminal,4
5,observational_aggregate,4
6,offshore_region,1
7,origin_terminal,6
8,pipeline,24
9,refinery,115


## 6. Variables (slots)

Two batches:

1. **Non-relational** — `production`, `consumption`, `inventory`, `balancing_item` for every node.
2. **Relational** — `inflow` / `outflow` pair per non-aggregation edge, deduped by `(source, target)`. Aggregation-type edges are skipped: they describe parent/child rollups, captured in `assets.attributes.resolution_hierarchy`.

Variable IDs follow the convention `<variable_type>__<commodity>__<node_id>` (non-relational) or `<variable_type>__<commodity>__<node_id>__<related_node_id>` (relational).

In [7]:
NON_RELATIONAL_TYPES = ("production", "consumption", "inventory", "balancing_item")

variable_rows = []

# 1. Non-relational slots — one per (node, type).
for n in nodes_json:
    for vt in NON_RELATIONAL_TYPES:
        variable_rows.append({
            "variable_id":     f"{vt}__{COMMODITY}__{n['id']}",
            "variable_type":   vt,
            "commodity":       COMMODITY,
            "node_id":         n["id"],
            "related_node_id": None,
            "description":     None,
            "attributes":      json.dumps({}),
        })

# 2. Relational slots — one inflow + one outflow per non-aggregation edge,
#    deduped by (source, target) so duplicate JSON edges (e.g. multiple
#    Enbridge routes) collapse into a single variable pair.
seen_pairs  = set()
agg_skipped = 0
for e in edges_json:
    if e["edge_type"] == "aggregation":
        agg_skipped += 1
        continue
    src, tgt = e["source"], e["target"]
    if (src, tgt) in seen_pairs:
        continue
    seen_pairs.add((src, tgt))

    edge_attr = json.dumps({
        "edge_type": e["edge_type"],
        **(e.get("attributes") or {}),
    })

    variable_rows.append({
        "variable_id":     f"outflow__{COMMODITY}__{src}__{tgt}",
        "variable_type":   "outflow",
        "commodity":       COMMODITY,
        "node_id":         src,
        "related_node_id": tgt,
        "description":     f"Outflow of {COMMODITY} from {src} to {tgt} (JSON edge_type={e['edge_type']})",
        "attributes":      edge_attr,
    })
    variable_rows.append({
        "variable_id":     f"inflow__{COMMODITY}__{tgt}__{src}",
        "variable_type":   "inflow",
        "commodity":       COMMODITY,
        "node_id":         tgt,
        "related_node_id": src,
        "description":     f"Inflow of {COMMODITY} into {tgt} from {src} (JSON edge_type={e['edge_type']})",
        "attributes":      edge_attr,
    })

with engine.begin() as c:
    c.execute(
        text("""
            INSERT INTO variables(variable_id, variable_type, commodity, node_id, related_node_id, description, attributes)
            VALUES (:variable_id, :variable_type, :commodity, :node_id, :related_node_id, :description, CAST(:attributes AS JSONB))
            ON CONFLICT (variable_id) DO UPDATE
              SET variable_type   = EXCLUDED.variable_type,
                  commodity       = EXCLUDED.commodity,
                  node_id         = EXCLUDED.node_id,
                  related_node_id = EXCLUDED.related_node_id,
                  description     = EXCLUDED.description,
                  attributes      = EXCLUDED.attributes
        """),
        variable_rows,
    )

print(f"Aggregation edges skipped: {agg_skipped}")
print(f"Variable rows written:     {len(variable_rows)}")
with engine.connect() as c:
    by_type = pd.read_sql(text("""
        SELECT variable_type, count(*) AS n FROM variables GROUP BY 1 ORDER BY 1
    """), c)
display(by_type)

Aggregation edges skipped: 0
Variable rows written:     1694


,variable_type,n
0,balancing_item,219
1,consumption,219
2,inflow,409
3,inventory,219
4,outflow,409
5,production,219


## 7. Scenario

Registers the starter scenario over `us_crude_starter`. The full coverage contract (authoritative levels, collapsed nodes, time range, notes) is preserved as `scenarios.attributes` JSONB so the eventual `variable_assignments` loader can consume it.

In [8]:
with engine.begin() as c:
    c.execute(
        text("""
            INSERT INTO scenarios(scenario_id, graph_id, name, description, attributes)
            VALUES (:scenario_id, :graph_id, :name, :description, CAST(:attributes AS JSONB))
            ON CONFLICT (scenario_id) DO UPDATE
              SET graph_id    = EXCLUDED.graph_id,
                  name        = EXCLUDED.name,
                  description = EXCLUDED.description,
                  attributes  = EXCLUDED.attributes
        """),
        {
            "scenario_id": SCENARIO_ID,
            "graph_id":    GRAPH_ID,
            "name":        contract_json.get("name"),
            "description": contract_json.get("description"),
            "attributes":  json.dumps({
                "time_range":           contract_json.get("time_range"),
                "authoritative_levels": contract_json.get("authoritative_levels"),
                "notes":                contract_json.get("notes"),
                "pipeline_layer_note":  contract_json.get("pipeline_layer_note"),
            }),
        },
    )

with engine.connect() as c:
    print("scenarios:", c.execute(text("SELECT count(*) FROM scenarios")).scalar())
    s = pd.read_sql(text("SELECT scenario_id, graph_id, name FROM scenarios ORDER BY scenario_id"), c)
display(s)

scenarios: 1


,scenario_id,graph_id,name
0,starter_us_crude_2015_2025,us_crude_starter,US_crude_starter_monthly_2015_2025


## 8. Promoted columns + `scenario_notes`

Populates the typed-column promotions and the single `scenario_notes` table created in Step 6 of the build notebook. Two things only:

- **8a. Promoted columns.** Walks each JSON node and writes the flat fields (`assets.node_class`, `assets.kind`, `assets.starter_status`, `nodes.starter_status`, `locations.spans_multiple_padds`) into typed columns. Writes `scenarios.time_range_start/end/frequency/pipeline_layer_note` from the contract. Writes `graphs.schema_version/thesis_version/source_file` from the JSON meta.
- **8f. scenario_notes.** Loads the JSON's `starter_coverage_contract.notes` array (free-form narrative). The earlier `node_scope_*` tables (`node_scopes`, `node_scope_authorisations`, `node_scope_collapsed_nodes`, `node_scope_notes`) are gone — authoritative levels and collapsed nodes are now exposed as views on top of `variable_assignments` (Step 7 of the build notebook), so loading them here would be redundant.

Idempotent. JSONB columns are kept untouched as a fallback / canonical reference.

In [9]:
# Helpers to coerce JSON values into typed Python values for parameterised SQL.
def _b(v):
    if v is None: return None
    return bool(v)

def _i(v):
    if v is None or v == "": return None
    try:    return int(v)
    except (TypeError, ValueError): return None

def _n(v):
    if v is None or v == "": return None
    try:    return float(v)
    except (TypeError, ValueError): return None

def parse_period(p):
    # "2015-01" -> "2015-01-01"; "2025-12" -> "2025-12-01"
    if p and len(p) == 7 and p[4] == "-":
        return f"{p}-01"
    return p

# ---------- 8a. Promote columns on existing tables ----------
with engine.begin() as c:
    # graphs
    c.execute(text("""
        UPDATE graphs SET
            schema_version = :sv,
            thesis_version = :tv,
            source_file    = :sf
        WHERE graph_id = :g
    """), {"g": GRAPH_ID,
           "sv": meta_json.get("schema_version"),
           "tv": meta_json.get("thesis_version"),
           "sf": str(JSON_PATH)})

    # locations.spans_multiple_padds
    for n in nodes_json:
        geog = n.get("geography") or {}
        loc_id = f"loc__{n['id']}"
        c.execute(text("""
            UPDATE locations SET spans_multiple_padds = :s WHERE location_id = :l
        """), {"l": loc_id, "s": _b(geog.get("spans_multiple_padds")) or False})

    # assets: node_class, node_subtype, kind, starter_status
    for n in nodes_json:
        cls = n.get("node_class")
        kind = "abstract" if cls == "observational" else "physical"
        c.execute(text("""
            UPDATE assets SET node_class=:c, node_subtype=:s, kind=:k, starter_status=:st
            WHERE asset_id = :a
        """), {"a": n["id"], "c": cls, "s": n["node_subtype"],
               "k": kind, "st": n.get("starter_status")})

    # nodes.starter_status
    for n in nodes_json:
        c.execute(text("UPDATE nodes SET starter_status=:s WHERE node_id=:n"),
                  {"n": n["id"], "s": n.get("starter_status")})

    # scenarios: time_range_start, time_range_end, frequency, pipeline_layer_note
    tr = (contract_json.get("time_range") or {})
    c.execute(text("""
        UPDATE scenarios SET
            time_range_start    = :s,
            time_range_end      = :e,
            frequency           = :f,
            pipeline_layer_note = :p
        WHERE scenario_id = :sc
    """), {"sc": SCENARIO_ID,
           "s": parse_period(tr.get("start")),
           "e": parse_period(tr.get("end")),
           "f": tr.get("frequency"),
           "p": contract_json.get("pipeline_layer_note")})

print("8a. Promoted columns updated on graphs / locations / assets / nodes / scenarios.")

# ---------- 8f. scenario_notes (the only piece of scope that doesn't derive) ----------
# Authoritative levels and collapsed nodes used to be loaded here too; they're
# now derived from variable_assignments via views v_scope_authoritative_nodes
# and v_scope_collapsed_nodes. Only the free-form notes array survives.
with engine.begin() as c:
    c.execute(text("DELETE FROM scenario_notes WHERE scenario_id = :s"),
              {"s": SCENARIO_ID})
    note_rows = [
        {"s": SCENARIO_ID, "p": i, "t": txt}
        for i, txt in enumerate(contract_json.get("notes") or [])
    ]
    if note_rows:
        c.execute(text("""
            INSERT INTO scenario_notes(scenario_id, position, note_text)
            VALUES (:s, :p, :t)
        """), note_rows)

print(f"8f. scenario_notes:                  {len(note_rows)} row(s)")

8a. Promoted columns updated on graphs / locations / assets / nodes / scenarios.


8f. scenario_notes:                  4 row(s)


## 8. Summary

In [10]:
with engine.connect() as c:
    inv = pd.read_sql(text("""
        SELECT 'commodities'                   AS relation, count(*) AS n FROM commodities
        UNION ALL SELECT 'graphs',                          count(*) FROM graphs
        UNION ALL SELECT 'locations',                       count(*) FROM locations
        UNION ALL SELECT 'assets',                          count(*) FROM assets
        UNION ALL SELECT '  assets (physical)',             count(*) FROM assets WHERE attributes->>'kind' = 'physical'
        UNION ALL SELECT '  assets (abstract)',             count(*) FROM assets WHERE attributes->>'kind' = 'abstract'
        UNION ALL SELECT 'node_types',                      count(*) FROM node_types
        UNION ALL SELECT 'nodes',                           count(*) FROM nodes
        UNION ALL SELECT 'variables',                       count(*) FROM variables
        UNION ALL SELECT '  variables (non-relational)',    count(*) FROM variables WHERE related_node_id IS NULL
        UNION ALL SELECT '  variables (relational)',        count(*) FROM variables WHERE related_node_id IS NOT NULL
        UNION ALL SELECT 'scenarios',                       count(*) FROM scenarios
        UNION ALL SELECT 'timeseries  (empty here)',        count(*) FROM timeseries
        UNION ALL SELECT 'timeseries_data  (empty here)',   count(*) FROM timeseries_data
        UNION ALL SELECT 'variable_assignments  (empty here)', count(*) FROM variable_assignments
    """), c)
display(inv)

print("\nLoad complete. The schema is structurally populated.")
print("timeseries, timeseries_data, and variable_assignments remain empty —")
print("those land in the next notebook when EIA series and the contract translation arrive.")

,relation,n
0,commodities,1
1,graphs,1
2,locations,219
3,assets,219
4,assets (physical),195
5,assets (abstract),24
6,node_types,19
7,nodes,219
8,variables,1694
9,variables (non-relational),876



Load complete. The schema is structurally populated.
timeseries, timeseries_data, and variable_assignments remain empty —
those land in the next notebook when EIA series and the contract translation arrive.
